# DSA 504 — Class 5
## pandas: Select, Filter, Sort, and Group

**Date:** Wednesday, Sep 16
**Reading:** *Python for Data Analysis*, ch. 8, 10
**Today's deliverable:** HW1 is assigned at the end of class (due before Class 6)

---

Today we start actually *asking questions* of `retail_sales.csv` — the same dataset from Class 4. Remember: this data still has the messiness we spotted last class (inconsistent category names, missing values, duplicates). We are **still not cleaning it today** — that's Classes 6–7. Today's tools work fine alongside messy data; we'll just get answers that are a little bit off because of it, which is itself a useful lesson in why cleaning matters.

### Learning goals
By the end of this class, you will be able to:
- Select one or more columns from a DataFrame
- Filter rows using boolean conditions, including combining multiple conditions
- Sort a DataFrame by one or more columns
- Group data and compute aggregate statistics per group
- Chain these operations together to answer real questions of the dataset


In [1]:
import pandas as pd
import numpy as np

sales = pd.read_csv("retail_sales.csv")
sales.head()


,date,store,category,units_sold,revenue
0,2025-04-22,Rome,Toys,25.0,402.50
1,2025-04-12,Syracuse,electronics,38.0,3154.00
2,2025-12-21,Utica,home goods,42.0,1299.06
3,2025-02-11,Syracuse,Electronics,60.0,5114.40
4,2025-04-10,Syracuse,toys,10.0,147.10


## 1. Selecting Columns

You've already seen single-column selection in Class 4. Today we go further: selecting multiple columns at once, and understanding what type of object each kind of selection returns.


In [2]:
# Selecting a single column -- returns a Series
revenue_column = sales["revenue"]
print(type(revenue_column))
revenue_column.head()


<class 'pandas.Series'>


0     402.50
1    3154.00
2    1299.06
3    5114.40
4     147.10
Name: revenue, dtype: float64

In [3]:
# Selecting multiple columns -- pass a LIST of column names, returns a DataFrame
subset = sales[["store", "category", "revenue"]]
print(type(subset))
subset.head()


<class 'pandas.DataFrame'>


,store,category,revenue
0,Rome,Toys,402.50
1,Syracuse,electronics,3154.00
2,Utica,home goods,1299.06
3,Syracuse,Electronics,5114.40
4,Syracuse,toys,147.10


**Common mistake:** `sales["store", "category"]` (no inner brackets) causes an error — you must pass a list, so it's double brackets: `sales[["store", "category"]]`.


In [4]:
# This is a common error students hit -- shown here on purpose, safely caught
try:
    broken = sales["store", "category"]
except KeyError as e:
    print(f"Error: {e}")
    print("Fix: use double brackets -- sales[['store', 'category']]")


Error: ('store', 'category')
Fix: use double brackets -- sales[['store', 'category']]


## 2. Filtering Rows (Boolean Indexing)

Filtering is how you answer questions like "show me only the rows where X." The mechanism is called **boolean indexing**: you write a condition that produces `True`/`False` for every row, and pandas keeps only the `True` ones.


In [5]:
# Step 1: a condition on its own produces a Series of True/False, one per row
is_high_revenue = sales["revenue"] > 3000
print(is_high_revenue.head())
print(type(is_high_revenue))


0    False
1     True
2    False
3     True
4    False
Name: revenue, dtype: bool
<class 'pandas.Series'>


In [6]:
# Step 2: put that condition INSIDE square brackets to filter the DataFrame
high_revenue_sales = sales[sales["revenue"] > 3000]
print(f"Rows before filtering: {len(sales)}")
print(f"Rows after filtering: {len(high_revenue_sales)}")
high_revenue_sales.head()


Rows before filtering: 4039
Rows after filtering: 588


,date,store,category,units_sold,revenue
1,2025-04-12,Syracuse,electronics,38.0,3154.00
3,2025-02-11,Syracuse,Electronics,60.0,5114.40
5,2025-05-07,Utica,Electronics,39.0,3531.06
24,2025-08-22,Syracuse,ELECTRONICS,38.0,3009.60
34,2025-02-11,Utica,Electronics,51.0,3674.55


### Filtering with multiple conditions

Use `&` for AND and `|` for OR — **not** the plain Python keywords `and`/`or`, which don't work correctly on pandas Series. Each condition needs its own parentheses.


In [7]:
# Multiple conditions with & (AND) -- both must be true
utica_electronics = sales[(sales["store"] == "Utica") & (sales["category"] == "Electronics")]
print(len(utica_electronics))
utica_electronics.head()


78


,date,store,category,units_sold,revenue
5,2025-05-07,Utica,Electronics,39.0,3531.06
34,2025-02-11,Utica,Electronics,51.0,3674.55
134,2025-11-07,Utica,Electronics,38.0,3244.44
165,2025-08-04,Utica,Electronics,63.0,5066.46
234,2025-12-17,Utica,Electronics,51.0,4177.92


In [8]:
# Multiple conditions with | (OR) -- either can be true
utica_or_rome = sales[(sales["store"] == "Utica") | (sales["store"] == "Rome")]
print(len(utica_or_rome))


2060


### Filtering with `.isin()`

When you want to match against several possible values in one column, `.isin()` is cleaner than chaining multiple `|` conditions.


In [9]:
# Instead of: (store == "Utica") | (store == "Rome") | (store == "Albany")
selected_stores = sales[sales["store"].isin(["Utica", "Rome", "Albany"])]
print(len(selected_stores))
selected_stores["store"].unique()


3056


<StringArray>
['Rome', 'Utica', 'Albany']
Length: 3, dtype: str

### Filtering with `.isna()` and `.notna()`

You already used `.isna().sum()` in Class 4 to *count* missing values. Today, use `.isna()` on its own to actually filter down to just the problem rows — useful for inspecting what's really going on before Classes 6–7 fix it.


In [10]:
# Rows where units_sold is missing
missing_units = sales[sales["units_sold"].isna()]
print(len(missing_units))
missing_units.head()


60


,date,store,category,units_sold,revenue
138,2025-12-18,Utica,Cloth ing,NaN,1130.92
155,2025-08-30,Utica,Home_Goods,NaN,1262.91
407,2025-06-29,Albany,Toys,NaN,711.00
446,2025-06-05,Albany,Groceries,NaN,690.42
469,2025-08-22,Utica,Clothing,NaN,1274.65


## 3. Sorting

`.sort_values()` reorders rows by one or more columns.


In [11]:
# Sort by a single column, ascending (default)
sales.sort_values("revenue").head()


,date,store,category,units_sold,revenue
3638,2025-07-31,Albany,Toys,7.0,114.10
1428,2025-05-16,Albany,Toys,8.0,121.20
312,2025-07-10,Syracuse,toys,9.0,123.66
1426,2025-06-05,Albany,toys,9.0,127.35
2017,2025-10-23,Syracuse,Home_Goods,4.0,133.56


In [12]:
# Sort descending -- highest revenue first
sales.sort_values("revenue", ascending=False).head()


,date,store,category,units_sold,revenue
1982,2025-05-17,Rome,ELECTRONICS,77.0,7021.63
69,2025-03-01,Utica,electronics,78.0,6814.08
3334,2025-05-18,Albany,Electronics,66.0,6395.40
3013,2025-04-05,Albany,electronics,76.0,6259.36
1241,2025-09-07,Utica,Electronics,70.0,6120.10


In [13]:
# Sort by multiple columns -- store first, then revenue within each store
sales.sort_values(["store", "revenue"], ascending=[True, False]).head(10)


,date,store,category,units_sold,revenue
3334,2025-05-18,Albany,Electronics,66.0,6395.40
3013,2025-04-05,Albany,electronics,76.0,6259.36
1467,2025-07-19,Albany,Electronics,78.0,6119.88
3740,2025-03-15,Albany,electronics,71.0,6049.20
532,2025-10-12,Albany,electronics,66.0,5866.08
618,2025-01-18,Albany,ELECTRONICS,64.0,5738.88
3143,2025-11-15,Albany,electronics,66.0,5697.78
1483,2025-08-12,Albany,Electronics,66.0,5697.12
2678,2025-01-25,Albany,electronics,66.0,5534.10
1740,2025-06-01,Albany,Electronics,61.0,5487.56


**Note:** `.sort_values()` returns a *new* sorted DataFrame by default — it doesn't change `sales` itself, the same way `sorted()` worked on lists back in Class 3. If you want to keep the sorted version, assign it to a variable.


## 4. Grouping — Aggregating by Category

`.groupby()` is one of the most useful tools in all of pandas. It answers questions like "what's the total revenue *per store*?" instead of one overall total.


In [14]:
# Total revenue per store
revenue_by_store = sales.groupby("store")["revenue"].sum()
print(revenue_by_store)
print(type(revenue_by_store))


store
Albany      1478141.38
Rome        1513342.88
Syracuse    1503175.92
Utica       1557200.64
Name: revenue, dtype: float64
<class 'pandas.Series'>


In [15]:
# Average units sold per category
avg_units_by_category = sales.groupby("category")["units_sold"].mean()
print(avg_units_by_category)


category
Cloth ing       64.338403
Clothing        63.767068
ELECTRONICS     44.167401
Electronics     42.838828
Groceries      130.081206
Home Goods      37.578947
Home_Goods      36.199262
Toys            25.901596
clothing        65.596610
electronics     44.384030
groceries      129.967089
home goods      38.818182
toys            26.068966
Name: units_sold, dtype: float64


**Talking point:** notice the category grouping above is split into far more groups than the 5 real categories — this is the `"Electronics"` vs `"electronics"` problem from Class 4 showing up concretely now. Grouping messy category data gives you *misleading* per-category totals, since what should be one group is scattered across several. This is exactly why cleaning happens before serious analysis.


In [16]:
# Grouping by MULTIPLE columns -- store AND category together
revenue_by_store_category = sales.groupby(["store", "category"])["revenue"].sum()
print(revenue_by_store_category.head(10))


store   category   
Albany  Cloth ing       98004.81
        Clothing       100251.59
        ELECTRONICS    191417.17
        Electronics    271509.46
        Groceries       89196.62
        Home Goods      67798.81
        Home_Goods      69472.98
        Toys            36018.32
        clothing       114890.85
        electronics    207733.37
Name: revenue, dtype: float64


### Multiple aggregations at once with `.agg()`

Often you want more than one statistic per group — `.agg()` lets you request several at once.


In [17]:
# Multiple statistics per store, in one call
summary = sales.groupby("store")["revenue"].agg(["sum", "mean", "count"])
print(summary)


                 sum         mean  count
store                                   
Albany    1478141.38  1493.072101    990
Rome      1513342.88  1493.921895   1013
Syracuse  1503175.92  1562.552931    962
Utica     1557200.64  1535.700828   1014


In [18]:
# .reset_index() turns the group labels back into a regular column,
# which is often useful before sorting or saving the result
summary_reset = summary.reset_index()
print(summary_reset)
print(type(summary_reset))


      store         sum         mean  count
0    Albany  1478141.38  1493.072101    990
1      Rome  1513342.88  1493.921895   1013
2  Syracuse  1503175.92  1562.552931    962
3     Utica  1557200.64  1535.700828   1014
<class 'pandas.DataFrame'>


## 5. Combining Filter, Group, and Sort

Real analysis usually chains several of today's tools together in one line.


In [19]:
# Question: which store had the highest total revenue, restricted to just electronics-labeled rows?
electronics_only = sales[sales["category"].str.lower() == "electronics"]
result = (
    electronics_only
    .groupby("store")["revenue"]
    .sum()
    .sort_values(ascending=False)
)
print(result)


store
Utica       748414.81
Syracuse    737686.60
Rome        680035.69
Albany      670660.00
Name: revenue, dtype: float64


**Note the `.str.lower()` trick above** — converting to lowercase before comparing/grouping is a quick (partial) workaround for the category-casing problem, letting `"Electronics"` and `"electronics"` count as the same thing for this one query. It doesn't fix the underlying data — the `ELECTRONICS` variant and the stray `"Cloth ing"` typo, for instance, still need real cleaning — but it's a useful preview of what's coming in Classes 6–7.


---
## Guided Practice

Work through these using `sales`, already loaded above.


### Exercise 1 — Select and filter
Select just the `store`, `date`, and `units_sold` columns, then filter that result down to rows where `units_sold` is greater than 100.


In [25]:
# Exercise 1 — your code here
# Select the specified columns using double brackets [[...]]
subset = sales[["store", "date", "units_sold"]]

# Filter rows where units_sold > 100
ex1_result = subset[subset["units_sold"] > 100]

ex1_result.head()


,store,date,units_sold
6,Rome,2025-03-04,129.0
8,Utica,2025-11-26,108.0
9,Rome,2025-09-04,135.0
22,Syracuse,2025-07-27,186.0
26,Utica,2025-05-28,151.0


### Exercise 2 — Multiple conditions
Filter the dataset to rows where the store is `"Rome"` **and** revenue is greater than 2000. How many rows match?


In [26]:
# Exercise 2 — your code here
# Enclose each condition in parentheses () and combine with &
rome_high_rev = sales[(sales["store"] == "Rome") & (sales["revenue"] > 2000)]

# Count matching rows
num_rows = len(rome_high_rev)

print(f"Number of matching rows: {num_rows}")
rome_high_rev.head()


Number of matching rows: 211


,date,store,category,units_sold,revenue
64,2025-08-03,Rome,ELECTRONICS,51.0,3954.03
84,2025-09-17,Rome,Electronics,45.0,3770.55
102,2025-10-19,Rome,Electronics,70.0,5440.40
106,2025-09-27,Rome,electronics,48.0,4152.48
116,2025-08-10,Rome,ELECTRONICS,57.0,4721.88


### Exercise 3 — Sort
Sort the dataset by `units_sold` in descending order, and print just the top 5 rows for the `store`, `category`, and `units_sold` columns.


In [27]:
# Exercise 3 — your code here
# Sort descending by units_sold, then select the specific columns and take top 5
top_units = sales.sort_values("units_sold", ascending=False)[["store", "category", "units_sold"]].head(5)

print(top_units)


         store   category  units_sold
838   Syracuse  Groceries       246.0
1790    Albany  Groceries       240.0
3434      Rome  groceries       228.0
3301     Utica  Groceries       227.0
3296    Albany  groceries       226.0


### Exercise 4 — Group
Compute total revenue per store. Which store has the highest total?


In [28]:
# Exercise 4 — your code here
# Group by store, select revenue, sum, and sort descending
store_revenue = sales.groupby("store")["revenue"].sum().sort_values(ascending=False)

print(store_revenue)

top_store = store_revenue.index[0]
top_value = store_revenue.iloc[0]
print(f"\nThe store with the highest total revenue is {top_store} (${top_value:,.2f}).")


store
Utica       1557200.64
Rome        1513342.88
Syracuse    1503175.92
Albany      1478141.38
Name: revenue, dtype: float64

The store with the highest total revenue is Utica ($1,557,200.64).


### Exercise 5 (stretch) — Combine everything
Filter to rows where `store` is `"Syracuse"`, group the result by `category`, compute both the sum and count of `revenue` per category using `.agg()`, and sort the result by summed revenue in descending order.


In [29]:
# Exercise 5 — your code here
syracuse_summary = (
    sales[sales["store"] == "Syracuse"]
    .groupby("category")["revenue"]
    .agg(["sum", "count"])
    .sort_values("sum", ascending=False)
)

print(syracuse_summary)


                   sum  count
category                     
electronics  297012.38     77
ELECTRONICS  242168.06     63
Electronics  198506.16     55
Clothing     106943.41     71
clothing     106806.55     68
Groceries     87860.34    104
Cloth ing     87605.12     56
Home Goods    80314.05     66
groceries     75928.55     89
Home_Goods    73797.41     65
home goods    72632.74     59
toys          40365.35    106
Toys          33235.80     83


---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


---
## Wrap-up

**Recap:** selecting one or more columns, filtering rows with boolean conditions (`&`, `|`, `.isin()`, `.isna()`), sorting with `.sort_values()`, and grouping with `.groupby()` plus `.agg()` for multiple statistics at once — and chaining all of these together in a single query.

**Common mistakes to watch for:**
- Using Python's `and`/`or` instead of `&`/`|` on pandas conditions
- Forgetting parentheses around each condition when combining them: `(a) & (b)`, not `a & b`
- Passing a single string instead of a list when selecting multiple columns: `sales[["store", "category"]]`, not `sales["store", "category"]`
- Forgetting that `.sort_values()` and `.groupby()` return *new* objects — they don't modify `sales` in place unless you reassign it

**HW1 is assigned today** — it builds directly on select/filter/sort/group, applied to this same dataset. Due before Class 6.

**Before next class (Sep 21):** Class 6 is Data Cleaning I — we finally fix the missing values and inconsistent categories we've been noticing (and working around) since Class 4.
